# LoopICL Classification Example

This notebook demonstrates how to use `LoopICLClassifier` on standard scikit-learn datasets by loading pre-trained weights from `results/weights/`.

## 1. Load Model from Checkpoint

In [ ]:
import numpy as np
from loopicl.interface import LoopICLClassifier

In [ ]:
# Load from the default checkpoint — config, weights, and device are all auto-detected.
# Pass weights_path="..." to use a different checkpoint.
clf = LoopICLClassifier()

n_params = sum(p.numel() for p in clf.model.parameters())
print(f"Loaded model — {n_params:,} parameters")

## 2. Evaluation Helper

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

def evaluate(name, X, y, test_size=0.3, random_state=42):
    """Split, fit, predict, and report metrics for a single dataset."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds)
    ll  = log_loss(y_test, proba)

    n_classes = proba.shape[1]
    if n_classes == 2:
        auc = roc_auc_score(y_test, proba[:, 1])
        auc_str = f"{auc:.4f}"
    else:
        auc = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
        auc_str = f"{auc:.4f} (OvR macro)"

    print(f"{name}")
    print(f"  train={len(y_train)}  test={len(y_test)}  classes={n_classes}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  ROC-AUC  : {auc_str}")
    print(f"  Log-loss : {ll:.4f}")
    print()
    return dict(dataset=name, accuracy=acc, roc_auc=auc, log_loss=ll)

## 3. Sklearn Datasets

### Iris (3-class, 150 samples)

In [ ]:
from sklearn.datasets import load_iris

data = load_iris()
results = [evaluate("Iris", data.data, data.target)]

## 4. Effect of Number of Loops

`num_loops` controls how many outer ICL iterations the model runs. More loops generally improve accuracy at the cost of compute. Fit once, sweep `num_loops` at predict time.

In [ ]:
import pandas as pd

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.3, random_state=42, stratify=data.target
)

clf.fit(X_train, y_train)

rows = []
for n in [1, 2, 3, 4, 5, 6]:
    proba = clf.predict_proba(X_test, num_loops=n)
    auc = roc_auc_score(y_test, proba, multi_class="ovr")
    rows.append({"num_loops": n, "ROC-AUC": round(auc, 4)})
    print(f"  num_loops={n}  ROC-AUC={auc:.4f}")

pd.DataFrame(rows).set_index("num_loops")

## 5. Early-Exit Inference

`use_early_exit=True` loads the early-exit checkpoint (`loopicl-cls-ee-step-4000.ckpt`).
The model learns a per-loop gate that decides when predictions have converged.
`exit_q` sets the CDF quantile threshold — lower values exit earlier (more speedup, slight accuracy trade-off).

| `exit_q` | Behaviour |
|----------|-----------|
| `1.0`    | Always run all loops (no early exit — reference) |
| `0.7`    | Conservative exit |
| `0.5`    | Balanced *(default)* |
| `0.3`    | Aggressive exit |

In [ ]:
import time

rows = []
for exit_q in [1.0, 0.9, 0.5, 0.1]:
    clf_ee = LoopICLClassifier(use_early_exit=True, exit_q=exit_q)
    clf_ee.fit(X_train, y_train)

    t0 = time.perf_counter()
    proba = clf_ee.predict_proba(X_test)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    auc = roc_auc_score(y_test, proba, multi_class="ovr")
    rows.append({"exit_q": exit_q, "ROC-AUC": round(auc, 4), "time_ms": round(elapsed_ms, 1)})
    print(f"  exit_q={exit_q}  ROC-AUC={auc:.4f}  ({elapsed_ms:.0f} ms)")

pd.DataFrame(rows).set_index("exit_q")